# 00 — Baseline inference smoke test

Load the **base** instruct model (`unsloth/Llama-3.2-3B-Instruct`) and run a few research-style earnings-call prompts.

**Kaggle**: enable GPU (T4). This notebook is a short smoke test only — no training.

Outputs here are the *before* snapshots for later base-vs-adapter comparison.

## 1. Install (Kaggle)

In [ ]:
# Detect Kaggle so we only pip-install on the hosted runtime.
from pathlib import Path
IN_KAGGLE = Path("/kaggle").exists()
print("IN_KAGGLE:", IN_KAGGLE)

In [ ]:
if IN_KAGGLE:
    %pip install -q unsloth transformers accelerate bitsandbytes pyyaml

## 2. Config

In [ ]:
from pathlib import Path

MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct"
MAX_SEQ_LENGTH = 2048
MAX_NEW_TOKENS = 256
LOAD_IN_4BIT = True

# Optional: overlay configs/default.yaml if the repo is cloned next to the notebook
cfg_path = Path("../configs/default.yaml")
if cfg_path.exists():
    import yaml
    with cfg_path.open() as f:
        cfg = yaml.safe_load(f)
    MODEL_NAME = cfg.get("model", {}).get("name", MODEL_NAME)
    MAX_SEQ_LENGTH = cfg.get("model", {}).get("max_seq_length", MAX_SEQ_LENGTH)
    LOAD_IN_4BIT = cfg.get("model", {}).get("load_in_4bit", LOAD_IN_4BIT)
    print("Loaded", cfg_path)
else:
    print("No local config found; using notebook defaults.")

print({"model": MODEL_NAME, "max_seq_length": MAX_SEQ_LENGTH, "load_in_4bit": LOAD_IN_4BIT})

## 3. Load base model

In [ ]:
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
)
FastLanguageModel.for_inference(model)
print("loaded", type(model).__name__)

## 4. Research-style smoke prompts

These are *ungrounded* prompts (no transcript attached). After Phase 1 we will add grounded versions that include source excerpts.

In [ ]:
SYSTEM = (
    "You are a financial research assistant. Answer clearly and conservatively. "
    "If the question cannot be answered from the provided context, say so. "
    "Do not invent numbers."
)

PROMPTS = [
    "Summarize the typical structure of a US large-cap quarterly earnings call (prepared remarks vs Q&A).",
    "An analyst asks: what questions should I listen for when management discusses gross margin vs operating margin?",
    "Define 'guide' vs 'consensus' vs 'beat/miss' in an earnings-call context, in two sentences each.",
]

def generate(user_text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user_text},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)
    with torch.inference_mode():
        out = model.generate(
            inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )
    new_tokens = out[0, inputs.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

for i, p in enumerate(PROMPTS, 1):
    print("=" * 72)
    print(f"[{i}] {p}")
    print("-" * 72)
    print(generate(p))
    print()

## 5. Done

If all three prompts returned coherent text, the baseline harness works.
Save this notebook output on Kaggle as the qualitative *base* snapshot.